In [3]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

print("Loading feature tables...")
train = pd.read_csv('train_features.csv')
test = pd.read_csv('test_features.csv')

drop_cols = ['id', 'label']
features = [c for c in train.columns if c not in drop_cols]

X = train[features]
y_string = train['label']
X_test = test[features]

# ---> Clean any infinite or missing values so XGBoost doesn't crash <---
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# Encode string labels to integers for XGBoost
labels_sorted = sorted(y_string.unique())
label_to_idx = {lab: i for i, lab in enumerate(labels_sorted)}
idx_to_label = {i: lab for i, lab in enumerate(labels_sorted)}
y = y_string.map(label_to_idx).values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 1. Train LightGBM Classifier
print("Training LightGBM...")
lgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.01,
    'max_depth': 7,
    'num_leaves': 31,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1
}
clf_lgb = lgb.LGBMClassifier(**lgb_params)
clf_lgb.fit(X_tr, y_tr)

# 2. Train XGBoost Classifier
print("Training XGBoost...")
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_tr), y=y_tr)
sample_weights = np.array([class_weights[label] for label in y_tr])

clf_xgb = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
clf_xgb.fit(X_tr, y_tr, sample_weight=sample_weights)

# 3. Blended Validation Prediction (Soft Voting via Probabilities)
print("Evaluating ensemble on validation set...")
lgb_probs = clf_lgb.predict_proba(X_val)
xgb_probs = clf_xgb.predict_proba(X_val)

# 50/50 Probability Blend
ensemble_probs = (lgb_probs + xgb_probs) / 2.0
val_preds = np.argmax(ensemble_probs, axis=1)

macro_f1 = f1_score(y_val, val_preds, average='macro')
print(f"\n---> Ensemble Validation Macro-F1 Score: {macro_f1:.4f} <---")

# 4. Retrain on 100% Data and Predict Test Set
print("\nRetraining models on 100% of data for final ensemble submission...")
clf_lgb.fit(X, y)

full_sample_weights = np.array([compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)[label] for label in y])
clf_xgb.fit(X, y, sample_weight=full_sample_weights)

print("Generating final test probabilities...")
test_lgb_probs = clf_lgb.predict_proba(X_test)
test_xgb_probs = clf_xgb.predict_proba(X_test)

final_test_probs = (test_lgb_probs + test_xgb_probs) / 2.0
final_preds_idx = np.argmax(final_test_probs, axis=1)

# Map back to string labels
final_preds_labels = [idx_to_label[idx] for idx in final_preds_idx]

submission = pd.DataFrame({
    'id': test['id'],
    'label': final_preds_labels
})
submission.to_csv('submission_ensemble.csv', index=False)
print("\nSaved 'submission_ensemble.csv'! Ready for final leaderboard upload. 🪐")

Loading feature tables...
Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000713 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4876
[LightGBM] [Info] Number of data points in the train set: 14400, number of used features: 22
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

In [4]:
import numpy as np
from sklearn.metrics import f1_score
from scipy.optimize import differential_evolution

# Ensure you have 'ensemble_probs', 'y_val', 'final_test_probs', and 'idx_to_label' ready from the previous script

n_classes = ensemble_probs.shape[1]

def objective_func(weights):
    """Calculates negative Macro-F1 for a given set of class probability weights."""
    weighted_probs = ensemble_probs * weights
    preds = np.argmax(weighted_probs, axis=1)
    # We use negative because scipy's optimizer minimizes functions (and we want to maximize F1)
    return -f1_score(y_val, preds, average='macro')

# Define bounds for each class weight (allowing them to scale between 0.1x and 5.0x)
bounds = [(0.1, 5.0) for _ in range(n_classes)]

print("Optimizing class probability weights for maximum Macro-F1...")
result = differential_evolution(objective_func, bounds, seed=42, maxiter=25, popsize=15)

best_weights = result.x
print(f"\nBest Class Weights Found: {best_weights}")

# Evaluate the optimized validation score
optimized_val_probs = ensemble_probs * best_weights
opt_val_preds = np.argmax(optimized_val_probs, axis=1)
opt_macro_f1 = f1_score(y_val, opt_val_preds, average='macro')
print(f"---> Optimized Validation Macro-F1 Score: {opt_macro_f1:.4f} <---")

# Apply these same weights to your test set probabilities to generate a stronger submission
print("\nApplying optimized weights to test predictions...")
final_test_probs_optimized = final_test_probs * best_weights
final_preds_idx_opt = np.argmax(final_test_probs_optimized, axis=1)

# Map back to string labels
final_preds_labels_opt = [idx_to_label[idx] for idx in final_preds_idx_opt]

submission_opt = pd.DataFrame({
    'id': test['id'],
    'label': final_preds_labels_opt
})
submission_opt.to_csv('submission_optimized_ensemble.csv', index=False)
print("Saved 'submission_optimized_ensemble.csv'! Ready for leaderboard upload. 🪐")

Optimizing class probability weights for maximum Macro-F1...

Best Class Weights Found: [2.48506734 2.869276   2.24044094 3.90870996 3.82666626 1.89641387
 4.35751809 4.5483273 ]
---> Optimized Validation Macro-F1 Score: 0.6191 <---

Applying optimized weights to test predictions...
Saved 'submission_optimized_ensemble.csv'! Ready for leaderboard upload. 🪐


In [5]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.utils.class_weight import compute_class_weight

# 1. Select High-Confidence Test Samples
confidence_threshold = 0.95
max_probs = np.max(final_test_probs, axis=1)
high_conf_mask = max_probs >= confidence_threshold

print(f"Found {high_conf_mask.sum()} high-confidence test samples out of {len(X_test)} to pseudo-label.")

if high_conf_mask.sum() > 0:
    # Extract high-confidence test features and their predicted labels
    X_test_high_conf = X_test[high_conf_mask].copy()
    y_test_high_conf = np.argmax(final_test_probs[high_conf_mask], axis=1)

    # 2. Augment the Original Training Data
    X_augmented = pd.concat([X, X_test_high_conf], ignore_index=True)
    y_augmented = np.concatenate([y, y_test_high_conf])

    print(f"Augmented training set size: {len(X_augmented)} samples (added {len(X_test_high_conf)} pseudo-labels).")

    # 3. Retrain LightGBM on Augmented Data
    print("\nRetraining LightGBM on augmented data...")
    lgb_params = {
        'n_estimators': 1500,
        'learning_rate': 0.01,
        'max_depth': 7,
        'num_leaves': 31,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }
    clf_lgb_pseudo = lgb.LGBMClassifier(**lgb_params)
    clf_lgb_pseudo.fit(X_augmented, y_augmented)

    # 4. Retrain XGBoost on Augmented Data
    print("Retraining XGBoost on augmented data...")
    aug_sample_weights = np.array([compute_class_weight(class_weight='balanced', classes=np.unique(y_augmented), y=y_augmented)[label] for label in y_augmented])
    
    clf_xgb_pseudo = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    clf_xgb_pseudo.fit(X_augmented, y_augmented, sample_weight=aug_sample_weights)

    # 5. Generate New Test Predictions with Pseudo-Labeled Ensemble
    print("Generating new test probabilities...")
    test_lgb_probs_p = clf_lgb_pseudo.predict_proba(X_test)
    test_xgb_probs_p = clf_xgb_pseudo.predict_proba(X_test)
    
    final_test_probs_pseudo = (test_lgb_probs_p + test_xgb_probs_p) / 2.0
    
    # Apply your previously optimized probability weights
    final_test_probs_pseudo_opt = final_test_probs_pseudo * best_weights
    final_preds_idx_pseudo = np.argmax(final_test_probs_pseudo_opt, axis=1)

    # Map back to string labels
    final_preds_labels_pseudo = [idx_to_label[idx] for idx in final_preds_idx_pseudo]

    submission_pseudo = pd.DataFrame({
        'id': test['id'],
        'label': final_preds_labels_pseudo
    })
    submission_pseudo.to_csv('submission_pseudolabeled.csv', index=False)
    print("\nSaved 'submission_pseudolabeled.csv'! Ready for leaderboard upload. 🪐")
else:
    print("No samples met the 0.95 confidence threshold. Try lowering it to 0.90.")

Found 894 high-confidence test samples out of 7000 to pseudo-label.
Augmented training set size: 18894 samples (added 894 pseudo-labels).

Retraining LightGBM on augmented data...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001439 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4879
[LightGBM] [Info] Number of data points in the train set: 18894, number of used features: 23
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Warning] No further spl

In [6]:
# Evaluate Pseudo-Labeled Model on your Validation Set (X_val, y_val)
print("Evaluating pseudo-labeled models on validation set...")

val_lgb_probs_p = clf_lgb_pseudo.predict_proba(X_val)
val_xgb_probs_p = clf_xgb_pseudo.predict_proba(X_val)

val_ensemble_probs_p = (val_lgb_probs_p + val_xgb_probs_p) / 2.0

# Apply your optimized weights from earlier
val_ensemble_probs_p_opt = val_ensemble_probs_p * best_weights
val_preds_p_opt = np.argmax(val_ensemble_probs_p_opt, axis=1)

pseudo_val_macro_f1 = f1_score(y_val, val_preds_p_opt, average='macro')
print(f"\n---> Pseudo-Labeled Validation Macro-F1 Score: {pseudo_val_macro_f1:.4f} <---")

Evaluating pseudo-labeled models on validation set...

---> Pseudo-Labeled Validation Macro-F1 Score: 0.8489 <---
